In [1]:
import pandas as pd
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn.conv import GATConv
from torch_geometric.nn import Linear
from torch_geometric.datasets import PPI
from torch import optim
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from tqdm import tqdm
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import f1_score

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [2]:
ppi_edges = pd.read_csv("syn_data/ppi_edge_index.csv")
ppi_x = pd.read_csv("syn_data/ppi_x.csv")
ppi_y = pd.read_csv("syn_data/ppi_y.csv")


In [3]:
ppi_data = PPI(root="./data")
val_data = PPI(root="./data", split="val")
test_data = PPI(root="./data", split="test")

In [4]:
ppi_data

PPI(20)

In [5]:
loader = DataLoader(
    ppi_data,
    batch_size=len(ppi_data)
)

val_loader = DataLoader(
    val_data,
    batch_size=len(val_data)
)
test_loader = DataLoader(
    test_data,
    batch_size=len(test_data)
)

In [6]:
batch_data = next(iter(loader))
val_batch_data = next(iter(val_data))
test_batch_data = next(iter(test_loader))

In [7]:
assert torch.allclose(torch.from_numpy(ppi_x.values).type(torch.float), batch_data.x), "Features does not match"
assert torch.allclose(torch.from_numpy(ppi_y.values).type(torch.float), batch_data.y), "Labels does not match"
assert torch.allclose(torch.from_numpy(ppi_edges.values).type(torch.long).transpose(-1, 0), batch_data.edge_index), "Edges does not match"

In [8]:
class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads, dropout, edge_dropout, num_layers, add_self_loops, residual):
        super(GNN, self).__init__()
        assert num_layers>=2, "Number of layers should be at least two"
        conv_0 = GATConv(in_dim, hidden_dim, heads, add_self_loops=add_self_loops, dropout=edge_dropout, residual = residual)
        self.convs = nn.ModuleList([conv_0])
        for i in range(num_layers-2):
            conv_i = GATConv(hidden_dim*heads, hidden_dim, heads, add_self_loops=add_self_loops, dropout = edge_dropout, residual = residual)
            lin_i = Linear(hidden_dim*heads, hidden_dim*heads)
            self.convs.append(conv_i)
        conv_out = GATConv(hidden_dim*heads, out_dim, dropout = edge_dropout, heads=heads, add_self_loops= add_self_loops, concat=False, residual = residual)
        self.convs.append(conv_out)
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self,x, edge_index):
        for i, layer in enumerate(self.convs):
            if i != (len(self.convs) - 1):
                x = self.dropout(x)
            x = layer(x, edge_index)
            if i != (len(self.convs) - 1):
                x = F.elu(x)
        return x

In [19]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
hyperparams = {
    "in_dim": ppi_x.shape[-1],
    "hidden_dim": 256,
    "out_dim": ppi_y.shape[-1],
    "heads": 4,                  # CHANGED: Reduced from 8 to 4
    "dropout": 0.0,
    "edge_dropout": 0.0,
    "add_self_loops": True,      # CHANGED: Enabled self-loops
    "num_layers": 3,
    "residual": True
}

epochs = 10000
PATIENCE = 100
patience = PATIENCE
model = GNN(**hyperparams).to(device)
optimizer = optim.Adam(model.parameters(), lr = 1e-3) # , weight_decay=5e-4
#optimizer = optim.SGD(model.parameters(), lr = 1e-3)
loss_fun = torch.nn.BCEWithLogitsLoss()
best_val_loss = torch.inf

In [20]:
def evaluate_loss(model, data):
    with torch.inference_mode():
        model.eval()
        logits = model(data.x, data.edge_index)
        return loss_fun(logits, data.y)

In [21]:

def evaluate(model, data):
    with torch.inference_mode():
        model.eval()
        logits = model(data.x, data.edge_index)
        preds = logits.cpu() > 0 
        return f1_score(data.y.cpu(), preds, average="micro")

In [22]:
batch_data = batch_data.to(device)
val_batch_data = val_batch_data.to(device)
test_batch_data = test_batch_data.to(device)

In [23]:
import copy
from tqdm import tqdm

# for epoch in tqdm(range(epochs)):
#     model.train()
#     logits = model(batch_data.x, batch_data.edge_index)
#     loss = loss_fun(logits, batch_data.y)
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
#     val_loss=evaluate_loss(model, val_batch_data)
#     patience -= 1
#     if best_val_loss > val_loss.item():
#         best_val_loss = val_loss.item()
#         patience = PATIENCE
#     if patience == 0:
#         print(f"Breaked at {epoch}")
#         break

In [24]:
evaluate(model, test_batch_data), evaluate(model, val_batch_data), evaluate(model, batch_data)

(0.38171710250005525, 0.3771975713764842, 0.38817828531008597)

In [25]:
# from CRUD import CRUD_Evaluator
# from PostgresList import PostgresList

# crud_evaluator = CRUD_Evaluator(PostgresList, "ppi_x.csv", "ppi_y.csv", "ppi_edge_index.csv", f"X_y_ppi_{PostgresList.file_suffix()}.csv")
# create_time = crud_evaluator.create()

In [26]:
subgraph_loader = NeighborLoader(
    batch_data.to(device),
    num_neighbors=[-1, -1, -1],  # All neighbors
    input_nodes=None,    # All nodes
    batch_size=1,     # Larger batch size for inference, if memory permits
    shuffle=False,
    num_workers=4
)

In [27]:
def reorder_pyg_data(data: Data, index: torch.Tensor) -> Data:
    """
    Reorders the nodes, features, labels, and edge_index of a PyG Data object.

    Args:
        data (Data): The input PyG Data object.
        index (torch.Tensor): A 1D tensor containing the new order of nodes.

    Returns:
        Data: A new Data object with reordered attributes.
    """
    # Create a deep copy to avoid modifying the original object
    new_data = data.clone().to(device)

    # Reorder all node-level attributes based on the provided index
    for key, value in data:
        if isinstance(value, torch.Tensor) and value.size(0) == data.num_nodes:
            new_data[key] = value.to(device)[index]

    # Create a mapping from old node indices to new node indices
    num_nodes = data.num_nodes
    # A tensor where the value at each old index is its new index
    remap_index = torch.zeros(num_nodes, dtype=torch.long, device = device)
    remap_index[index] = torch.arange(num_nodes).to(device)

    # Apply the mapping to the edge_index
    new_data.edge_index = remap_index[data.edge_index]


    return new_data

In [28]:
best_model = None

In [29]:
from torch_geometric.utils import sort_edge_index
import copy

model.train()
model = model.to(device)
accumulation_steps = 5
for epoch in range(100):
    acc_loss = 0
    for i, batch in enumerate(tqdm(subgraph_loader)):
        batch = batch.to(device)
        batch = reorder_pyg_data(batch, torch.argsort(batch.n_id).to(device))
        mask = (batch.n_id == batch.input_id).to(device)
        logits = model(batch.x, batch.edge_index)
        loss = loss_fun(logits[mask], batch.y[mask])
        
        loss.backward()
        
        batch = batch.cpu()
        loss = evaluate_loss(model, val_batch_data)
        acc_loss += loss.item()
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()        # update weights
            optimizer.zero_grad()
    patience -= 1
    if best_val_loss > acc_loss:
        best_val_loss = acc_loss
        patience = PATIENCE
        best_model = copy.deepcopy(model)
    if patience == 9:
        print(f"Breaked at {epoch}")
        break

 24%|█████████████████▉                                                         | 10733/44906 [31:28<1:40:13,  5.68it/s]
Exception ignored in: <object repr() failed>Exception ignored in sys.unraisablehook: <built-in function unraisablehook>
KeyboardInterrupt: 


KeyboardInterrupt: 

In [30]:
evaluate(model, test_batch_data.to(device)), evaluate(model, val_batch_data.to(device)), evaluate(model, batch_data.to(device))

(0.9585022426194391, 0.9432240519715969, 0.9689553122753439)

In [62]:
from torch_geometric.utils import sort_edge_index
for i, batch in enumerate(tqdm(subgraph_loader)):
    seed_id = i
    _, subgraph = crud_evaluator.read_subgraph_for_training(seed_id, 3)
    if subgraph is None: continue
    mask = subgraph.node_ids == seed_id
    data = Data(x = torch.from_numpy(subgraph.features).type(torch.float), y = torch.from_numpy(subgraph.labels).type(torch.float), edge_index = torch.from_numpy(subgraph.edge_index).type(torch.long), mask = torch.from_numpy(mask).type(torch.bool))

    batch = reorder_pyg_data(batch, torch.argsort(batch.n_id).to(device))
    features_match = torch.allclose(batch.x.cpu() , data.x)
    labels_match = torch.allclose(batch.y.cpu() , data.y)
    edges_match = torch.allclose(sort_edge_index(batch.edge_index.cpu()) , sort_edge_index(data.edge_index))
    assert features_match, "The node features do not match."
    assert labels_match, "The node labels do not match."
    assert edges_match, "The graph edges do not match."
    # model.train()
    # logits = model(batch.x, batch.edge_index)
    # loss = loss_fun(logits[batch.mask], data.y[batch.mask])
    # optimizer.zero_grad()
    # loss.backward()
    # optimizer.step()
    # data = data.cpu()

 13%|██████████▎                                                                 | 6058/44906 [21:42<2:19:11,  4.65it/s]


KeyboardInterrupt: 

In [18]:
for i in tqdm(range(ppi_x.shape[0])):
    seed_id = i
    _, subgraph = crud_evaluator.read_subgraph_for_training(seed_id, 3)
    if subgraph is None: continue
    mask = subgraph.node_ids == seed_id
    data = Data(x = torch.from_numpy(subgraph.features).type(torch.float), y = torch.from_numpy(subgraph.labels).type(torch.float), edge_index = torch.from_numpy(subgraph.edge_index).type(torch.long), mask = torch.from_numpy(mask).type(torch.bool))
    data = data.to(device)
    model.train()
    logits = model(data.x, data.edge_index)
    loss = loss_fun(logits[data.mask], data.y[data.mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    data = data.cpu()

  1%|▊                                                                            | 457/44906 [01:34<2:33:12,  4.84it/s]


KeyboardInterrupt: 

In [16]:
delete_time = crud_evaluator.delete()

Delete


AttributeError: 'NoneType' object has no attribute 'close'